In [ ]:
-- Анализ стабильности номиналов по месяцам
-- Проверяем, какая доля клиентов сохраняет тот же номинал
-- относительно предыдущего месяца / флайта

with prepared as (

    -- Подготовка данных:
    -- извлекаем числовой номинал из текстового описания сегмента

    select
        client_id,
        campaign_name,
        date_trunc('month', mrc_start_date)::date as month_dt,

        case
            when lower(com_cus_sgr_desc) like '%200%' then 200
            when lower(com_cus_sgr_desc) like '%300%' then 300
            when lower(com_cus_sgr_desc) like '%400%' then 400
            when lower(com_cus_sgr_desc) like '%500%' then 500
            else null
        end as nominal_amount

    from cvm_sbx.YOUR_TABLE

    where client_id is not null
),

client_nominal_history as (

    -- Для каждого клиента:
    -- сортируем кампании по времени
    -- и получаем предыдущий номинал

    select
        client_id,
        campaign_name,
        month_dt,
        nominal_amount,

        lag(nominal_amount) over (
            partition by client_id
            order by month_dt
        ) as prev_nominal

    from prepared

    where nominal_amount is not null
),

nominal_comparison as (

    -- Сравниваем текущий и предыдущий номинал

    select
        *,
        case
            when nominal_amount = prev_nominal then 1
            else 0
        end as same_nominal_flag

    from client_nominal_history

    where prev_nominal is not null
)

-- Финальная агрегация по месяцам

select
    month_dt,

    count(*) as transitions_cnt,

    sum(same_nominal_flag) as same_nominal_cnt,

    round(
        sum(same_nominal_flag) * 100.0 /
        count(*),
        2
    ) as same_nominal_pct

from nominal_comparison

group by month_dt

order by month_dt;

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

df_nominal['month_dt'] = pd.to_datetime(df_nominal['month_dt'])

plt.figure(figsize=(10, 5))

plt.plot(
    df_nominal['month_dt'],
    df_nominal['same_nominal_pct'],
    marker='o'
)

plt.title('Доля клиентов с тем же номиналом')
plt.xlabel('Месяц')
plt.ylabel('Same nominal, %')

ax = plt.gca()

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()